In [0]:
bcrpdata_root = (
    "abfss://bronze@starlpdevdiego01.dfs.core.windows.net/"
    "bcrpdata/"
)

display(dbutils.fs.ls(bcrpdata_root))

In [0]:
bronze_root = (
    "abfss://bronze@starlpdevdiego01.dfs.core.windows.net/"
    "bcrpdata/exchange_rates/series_code=pd04638pd/"
    "ingestion_date=2026-10-11/"
    "run_id=e7d0e002-b3a6-4de3-8157-c2a3f818cd38"
)

display(dbutils.fs.ls(bronze_root))

In [0]:
import importlib
import transformations.silver.bcrpdata_transform as bcrpdata

importlib.reload(bcrpdata)

observations = bcrpdata.read_bcrpdata_observations(
    spark,
    f"{bronze_root}/response.json",
)

total = observations.count()
unavailable = observations.filter("raw_value = 'n.d.'").count()

print("Observaciones:", total)
print("No disponibles:", unavailable)

assert total == 195, "Se esperaban 195 observaciones"
assert unavailable == 11, "Se esperaban 11 valores n.d."

display(observations.limit(5))

In [0]:
import json
from datetime import date, datetime, timezone
from uuid import uuid4

metadata = json.loads(
    dbutils.fs.head(
        f"{bronze_root}/response.json.metadata.json",
        65536,
    )
)

extracted_at = datetime.fromisoformat(
    metadata["extracted_at"].replace("Z", "+00:00")
)

start_date = date(2026, 1, 1)
end_date = date(2026, 9, 30)
ingestion_run_id = "e7d0e002-b3a6-4de3-8157-c2a3f818cd38"
processing_run_id = str(uuid4())
processed_at = datetime.now(timezone.utc)

normalized = bcrpdata.normalize_bcrpdata_observations(
    observations,
    "PD04638PD",
    start_date,
    end_date,
    ingestion_run_id,
    processing_run_id,
    extracted_at,
    processed_at,
)

checks = bcrpdata.validate_bcrpdata_observations(
    normalized,
    start_date,
    end_date,
)

print("CONTROLES:", checks)

normalized.groupBy("available").count().show()

display(
    normalized.select(
        "series_code",
        "observation_date",
        "raw_value",
        "rate_pen_per_usd",
        "available",
    ).orderBy("observation_date").limit(5)
)

In [0]:
silver_root = (
    "abfss://silver@starlpdevdiego01.dfs.core.windows.net"
)

result = bcrpdata.write_bcrpdata_delta(
    spark,
    normalized,
    silver_root,
)

print("RESULTADO:", result)
print("Processing run ID:", processing_run_id)

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS
    dbw_arlp_dev_diego01.silver_reference.exchange_rates
    USING DELTA
    LOCATION 'abfss://silver@starlpdevdiego01.dfs.core.windows.net/reference/exchange_rates'
""")

spark.sql("""
    SELECT available, COUNT(*) AS observations
    FROM dbw_arlp_dev_diego01.silver_reference.exchange_rates
    GROUP BY available
""").show()